# PSF Defocus Z-Scan Analysis

Vectorial Debye PSF simulation sweep for **ATTO 488**, **ATTO 565**, and **ATTO 647N**.  
Quantifies the effect of axial defocus on localisation precision and colour precision  
using the standard S3M fitting pipeline (`FittingStrategy.STANDARD`).

Physics model: vectorial Debye + Gibson–Lanni spherical aberration (NA 1.49, oil/water).

**Structure**
1. PSF shape visualisation — per-channel (B/G/R) RGB composite at increasing defocus  
2. Z-sweep simulation — 7 z-offsets × 2 coverslip depths × 3 photon levels  
3. Analysis — localisation precision and colour precision vs defocus

In [1]:
import sys, os, glob, types
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import polars as pl

sys.path.append(str(Path(os.getcwd()).parent.parent))

from src import IOFunctions, SpectralFunctions, MaskFunctions, sCMOSFunctions, PlottingBase
from src.Multicolour_Simulation_Functions import (
    FittingStrategy, SimulationConfig, MultiC_Sim_Funcs,
)
from src.simulation.defocus_psf import VectorialPSF

IO    = IOFunctions.IO_Functions()
S_F   = SpectralFunctions.Spectral_Funcs()
M_F   = MaskFunctions.Mask_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MSF   = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter()

In [2]:
# ── Camera calibration ────────────────────────────────────────────────────────
cal_folder = Path("../../Camera_Calibrations/Ximea_Camera/")
gain      = IO.read_tiff(str(cal_folder / "gain.tif"))
offset    = IO.read_tiff(str(cal_folder / "offset.tif"))
variance  = IO.read_tiff(str(cal_folder / "variance.tif"))
readnoise = float(np.median(IO.read_tiff(str(cal_folder / "readnoise.tif"))))
rqe       = IO.read_tiff(str(cal_folder / "rqe.tif"))

print(f"Camera calibration loaded — sensor: {gain.shape}, readnoise: {readnoise:.2f}")

Camera calibration loaded — sensor: (1544, 2064), readnoise: 2.34


In [12]:
# ── Spectral setup ────────────────────────────────────────────────────────────
R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])   # (3, n_wl); convention B=0, G=1, R=2

# Dyes and global simulation constants
dyes          = ["ATTO 488", "ATTO 565", "ATTO 647N"]
filters       = []          # no bandpass filter
NA            = 1.49
pixel_size_nm = 69.0        # Ximea object-space pixel (nm)
image_dims    = 32          # 22 × 22 px  ≈  1.52 µm FOV
background_photons = 40.0

# Smoothing function (Gaussian, σ = 1.5 px) ─ must be defined before MSF calls
smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5},
    extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack,
    data_arg="image",
)

# Coarser wavelength grid for PSF integration (5 nm steps; sufficient to <1% PSF error)
wl_psf_nm = np.arange(400, 751, 5, dtype=float)

# build_spectral_weights requires pixel_QYs on the same grid as wl_psf_nm;
# the native grid from getpixelefficiency() may differ — interpolate.
pixel_QYs_psf = np.vstack([
    np.interp(wl_psf_nm, wavelength, pixel_QYs[c])
    for c in range(pixel_QYs.shape[0])
])  # (3, len(wl_psf_nm))

print(f"Wavelength grid: {wavelength[0]:.0f}–{wavelength[-1]:.0f} nm  "
      f"({len(wavelength)} pts);  PSF grid: {len(wl_psf_nm)} pts @ 5 nm steps")

Wavelength grid: 400–999 nm  (600 pts);  PSF grid: 71 pts @ 5 nm steps


In [13]:
# ── Save folder ───────────────────────────────────────────────────────────────
save_folder = Path("/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/Z_Position_Effect")
save_folder.mkdir(parents=True, exist_ok=True)
print(f"Results → {save_folder}")

Results → /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/Z_Position_Effect


In [14]:
# ── VectorialPSF objects and per-dye spectral weights ─────────────────────────
# N_pupil=128 is fast enough for the display figure; simulation dispatch uses 256
vpsf_display = VectorialPSF(
    NA=NA, n_medium=1.33, n_immersion=1.515,
    pix_obj_um=pixel_size_nm / 1000.0,
    psf_size=21, N_pupil=256,
)

# Spectral weights: w_c(λ) = S(λ)·T_obj(λ)·QE_c(λ), shape (n_wl_psf, 3) = (n_wl_psf, B/G/R)
# pixel_QYs_psf is already interpolated to wl_psf_nm in the spectral setup cell.
spectral_weights = {}
channel_fractions = {}
for dye in dyes:
    w = VectorialPSF.build_spectral_weights(
        spectral_functions=S_F,
        dye=dye,
        filters=None,
        wavelengths_nm=wl_psf_nm,
        pixel_QYs=pixel_QYs_psf,   # (3, n_wl_psf) — matched grid
        include_objective=True,
    )  # (n_wl_psf, 3)
    spectral_weights[dye] = w
    frac = w.sum(axis=0) / w.sum()
    channel_fractions[dye] = frac
    print(f"{dye:12s}  B={frac[0]:.3f}  G={frac[1]:.3f}  R={frac[2]:.3f}")

ATTO 488      B=0.143  G=0.745  R=0.112
ATTO 565      B=0.027  G=0.392  R=0.581
ATTO 647N     B=0.058  G=0.203  R=0.739


## PSF Shape Visualisation

Polychromatic vectorial PSF at each defocus step, integrated over the dye emission  
spectrum weighted by per-channel Bayer QE.  Each panel is an **RGB composite**:  
B-channel PSF → blue, G-channel PSF → green, R-channel PSF → red.  
Intensities are normalised to the peak of the **in-focus** PSF for that channel,  
so dimming of the central peak with defocus is visible.

In [46]:
# ── Compute PSF patches for visualisation ────────────────────────────────────
z_display_nm = np.array([0, 100, 200, 300, 500], dtype=float)
z_display_um = z_display_nm / 1000.0
wl_display_um = wl_psf_nm / 1000.0

psf_display = {}    # psf_display[dye] shape: (n_z, 3, psf_size, psf_size)
for dye in dyes:
    psf_display[dye] = vpsf_display.compute_psf_stack(
        z_offsets_um=z_display_um,
        wavelengths_um=wl_display_um,
        spectral_weights=spectral_weights[dye],
        distance_from_coverslip_um=0.0,
    )  # (n_z, 3, 21, 21)
    print(f"{dye} PSF stack: {psf_display[dye].shape}  "
          f"in-focus B/G/R peak: {psf_display[dye][0].max(axis=(-2,-1))}")

ATTO 488 PSF stack: (5, 3, 21, 21)  in-focus B/G/R peak: [ 0.10202751  0.09785394  0.08564817]
ATTO 565 PSF stack: (5, 3, 21, 21)  in-focus B/G/R peak: [ 0.07539308  0.07976221  0.07517223]
ATTO 647N PSF stack: (5, 3, 21, 21)  in-focus B/G/R peak: [ 0.06195435  0.06117706  0.06259334]


In [51]:
# ── Figure 1: raw vectorial PSF at increasing defocus ────────────────────────
# |FFT(E)|² from the Debye model: no noise, no camera QE, no Bayer mask.
# 25 nm/pixel oversamples the Airy disk (~9 px across) to show ring structure.
# Each panel normalised to its own peak to reveal structure at every defocus.

pix_raw_nm   = 5.0
psf_size_raw = int(3000/5)          # 81 × 25 nm = 2.025 µm window

vpsf_raw = VectorialPSF(
    NA=NA, n_medium=1.33, n_immersion=1.515,
    pix_obj_um=pix_raw_nm / 1000.0,
    psf_size=psf_size_raw,
    N_pupil=512,
)

z_disp_raw_nm = np.array([0, 100, 250, 500], dtype=float)

psf_raw_stack = vpsf_raw.compute_psf_stack(
    z_offsets_um=z_disp_raw_nm / 1000.0,
    wavelengths_um=np.array([0.55]),
    spectral_weights=None,
    distance_from_coverslip_um=0.0,
)  # (n_z, 1, ps, ps)

n_z_raw = len(z_disp_raw_nm)

fig, axs = plotter.two_column_plot(nrows=1, ncols=n_z_raw, height=6.69 / n_z_raw + 0.05)
fs = plotter.config.font_size

for i_z in range(n_z_raw):
    I = psf_raw_stack[i_z, 0]
    I_disp = I / I.max()
    I_log = np.log10(np.maximum(I_disp, 1e-4))   # floor avoids log(0)
    plotter.image_plot(
        axs[i_z], I_log,
        cmap="gray",
        vmin=np.percentile(I_log, 0.1), vmax=np.percentile(I_log, 99.9),
        scalebar=True,
        pixelsize=pix_raw_nm,
        scalebarsize=500.0,
        scalebarlabel="500 nm",
        scalebar_color="white",
        origin="lower",
        colorbar=(i_z == 3)
    )

    # In-panel annotation: defocus label in top-left corner
    label = f"Δz = {z_disp_raw_nm[i_z]:.0f} nm"
    axs[i_z].text(
        0.05, 0.97, label,
        transform=axs[i_z].transAxes,
        fontsize=fs, color="white",
        va="top", ha="left",
        bbox=dict(boxstyle="square,pad=0.15", fc="none", ec="none"),
    )

plotter.save_or_show(fig, save_path=str(save_folder / "psf_defocus_raw.svg"), show=False)

INFO:src.PlottingBase:Plot saved to: /home/jbeckwith/Documents/Chemistry/Lee/Data/Simulation/20260623_DefocusZScan/psf_defocus_raw.svg


## Z-Sweep Simulation

For each (dye, z-offset, coverslip depth), run `test_simulation_method` with  
`FittingStrategy.STANDARD` across three photon levels.  
Two coverslip depths probe the effect of depth-induced spherical aberration:
- **0 nm** — emitter at the coverslip (no spherical aberration)  
- **500 nm** — emitter 500 nm into the aqueous sample (realistic cell imaging)

In [15]:
# ── Sweep parameters ─────────────────────────────────────────────────────────
z_offsets_nm       = np.array([0, 50, 100, 150, 200, 300, 400, 500], dtype=float)
coverslip_depths_nm = np.array([0, 500], dtype=float)
n_photon_space     = np.geomspace(500, 20000, 100)

# n_bootstrap: number of simulated localisations per (z, photon, dye, d_coverslip).
# 2000 gives ~3% RMSE uncertainty (1/√2000); increase to 10 000 for publication quality.
n_bootstrap = 2000

print(f"Sweep:  {len(dyes)} dyes  ×  {len(z_offsets_nm)} z-offsets  "
      f"×  {len(coverslip_depths_nm)} coverslip depths  "
      f"×  {len(n_photon_space)} photon levels")
print(f"Total simulation calls: "
      f"{len(dyes) * len(z_offsets_nm) * len(coverslip_depths_nm)}")

Sweep:  3 dyes  ×  8 z-offsets  ×  2 coverslip depths  ×  100 photon levels
Total simulation calls: 48


In [16]:
# ── Camera parameters dictionary (constant across sweep) ─────────────────────
def make_camera_params(image_dims):
    """Median-flat camera calibration for a square image of size image_dims."""
    return {
        "gain":               np.full((image_dims, image_dims), np.median(gain)),
        "offset":             np.full((image_dims, image_dims), np.median(offset)),
        "variance":           np.full((image_dims, image_dims), np.median(variance)),
        "readnoise":          readnoise,
        "rqe":                np.full((image_dims, image_dims), np.median(rqe)),
        "masks":              M_F.get_masks(size_x=image_dims, size_y=image_dims),
        "pixel_QYs":          pixel_QYs,
        "pixel_order":        ["B", "G", "R"],
        "pixel_order_indices": {"B": 0, "G": 1, "R": 2},
    }

camera_params_dict = make_camera_params(image_dims)
print(f"Camera parameters dict ready — image size: {image_dims}×{image_dims} px")

Camera parameters dict ready — image size: 32×32 px


In [ ]:
# ── Run sweep ─────────────────────────────────────────────────────────────────
import time

n_total  = len(dyes) * len(z_offsets_nm) * len(coverslip_depths_nm)
i_run    = 0
t_start  = time.time()

for dye in dyes:
    for z_nm in z_offsets_nm:
        for d_nm in coverslip_depths_nm:
            i_run += 1
            flag = f"defocus_z{z_nm:04.0f}nm_d{d_nm:04.0f}nm_"

            config = SimulationConfig(
                n_bootstrap=n_bootstrap,
                background_photons=background_photons,
                NA=NA,
                pixel_size=pixel_size_nm,
                save_raw_results=True,
                subtractx0y0=False,    # keep absolute positions; residuals computed in analysis
                use_stochastic_photons=True,
                save_summary_csvs=True,
                verbose=False,
                defocus_z_um=z_nm / 1000.0,
                distance_from_coverslip_um=d_nm / 1000.0,
            )

            print(f"[{i_run:3d}/{n_total}]  {dye}  z={z_nm:4.0f} nm  "
                  f"d={d_nm:4.0f} nm", end="  ", flush=True)

            MSF.test_simulation_method(
                dye=dye,
                filters=filters,
                wavelength=wavelength,
                camera_parameters=camera_params_dict,
                save_folder=str(save_folder),
                n_photon_space=n_photon_space,
                smoothing_function=smoothing_function,
                strategy=FittingStrategy.STANDARD,
                starting_flag=flag,
                config=config,
                overwrite=True,
            )

            elapsed = (time.time() - t_start) / 60.0
            rate    = i_run / elapsed if elapsed > 0 else 0
            eta     = (n_total - i_run) / rate if rate > 0 else 0
            print(f"done   ({elapsed:.1f} min elapsed, ETA {eta:.1f} min)")

print(f"\nSweep complete in {(time.time()-t_start)/60:.1f} min.")

[  1/48]  ATTO 488  z=   0 nm  d=   0 nm  

INFO:simulation.multicolour:Overwrite=True: Deleting existing results file: defocus_z0000nm_d0000nm_LM_method_ATTO 488_rawresults.h5
/home/jsb92/Documents/venv/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


INFO:simulation.multicolour:
Completed analysis of 100 photon flux values    Total time: 6.612 min
INFO:simulation.multicolour:Simulation completed for strategy standard


done   (6.6 min elapsed, ETA 310.9 min)


## Analysis

Load the RMSE summary CSVs saved by `test_simulation_method` and aggregate into  
an xarray for plotting.  

Metrics (from `_compute_fit_statistics`):
- **`xc`, `yc`** — RMSE of fitted position in nm  
- **`colour_distance`** — mean Euclidean distance of fitted (A_B, A_G, A_R) from truth

In [ ]:
import xarray as xr
import matplotlib.colors as mcolors

# ── Build xarray DataArray from RMSE summary CSVs ─────────────────────────────
# Dims: [dye, z_nm, coverslip_nm, n_photons, metric]
metrics = ["sigma_xy", "colour_dist"]

da_defocus = xr.DataArray(
    data=np.full(
        [len(dyes), len(z_offsets_nm), len(coverslip_depths_nm),
         len(n_photon_space), len(metrics)],
        np.nan,
    ),
    coords={
        "dye":          dyes,
        "z_nm":         z_offsets_nm,
        "coverslip_nm": coverslip_depths_nm,
        "n_photons":    n_photon_space,
        "metric":       metrics,
    },
    dims=["dye", "z_nm", "coverslip_nm", "n_photons", "metric"],
)

missing = []
for i_dye, dye in enumerate(dyes):
    dye_str = dye.replace("/", "-")
    for i_z, z_nm in enumerate(z_offsets_nm):
        for i_d, d_nm in enumerate(coverslip_depths_nm):
            flag = f"defocus_z{z_nm:04.0f}nm_d{d_nm:04.0f}nm_"
            pattern = str(save_folder / f"{flag}*{dye_str}*RMSE_mean*.csv")
            files = sorted(glob.glob(pattern))
            if not files:
                missing.append((dye, z_nm, d_nm))
                continue
            df = pl.read_csv(files[0])
            for i_ph, row in enumerate(df.iter_rows(named=True)):
                da_defocus[i_dye, i_z, i_d, i_ph, 0] = float(
                    np.sqrt((row["xc"] ** 2 + row["yc"] ** 2) / 2)
                )
                da_defocus[i_dye, i_z, i_d, i_ph, 1] = float(row["colour_distance"])

if missing:
    print(f"Missing files: {missing}")
da_defocus.to_netcdf(str(save_folder / "defocus_sweep.nc"))
print(f"DataArray shape: {da_defocus.shape}")
print(f"σ_xy  range : {float(da_defocus.sel(metric='sigma_xy').min()):.1f}–"
      f"{float(da_defocus.sel(metric='sigma_xy').max()):.1f} nm")
print(f"colour range: {float(da_defocus.sel(metric='colour_dist').min()):.4f}–"
      f"{float(da_defocus.sel(metric='colour_dist').max()):.4f}")

In [ ]:
# ── Plotting helpers ──────────────────────────────────────────────────────────
dye_labels   = {"ATTO 488": "ATTO 488", "ATTO 565": "ATTO 565", "ATTO 647N": "ATTO 647N"}
depth_labels = {0.0: "d = 0 nm (coverslip)", 500.0: "d = 500 nm (into sample)"}
fs = plotter.config.font_size

In [ ]:
# ── Figure 2: localisation precision σ_xy — surface map ──────────────────────
# X = defocus (nm), Y = photon count (log scale), colour = σ_xy (nm, viridis).
# Layout: 2 rows (coverslip depth) × 3 cols (dye); shared colour scale per figure.
metric     = "sigma_xy"
cbar_label = r"$\sigma_{xy}$ / nm"

data_m = da_defocus.sel(metric=metric).values  # (n_dye, n_z, n_d, n_ph)
vmin, vmax = float(np.nanmin(data_m)), float(np.nanmax(data_m))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(
    nrows=len(coverslip_depths_nm), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
    squeeze=False,
)
for i_d, d_nm in enumerate(coverslip_depths_nm):
    for i_c, dye in enumerate(dyes):
        ax = axs[i_d, i_c]
        # da shape at this point: (n_z, n_photons); transpose → (n_photons, n_z) for pcolormesh
        Z = da_defocus.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel(
            f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "",
            fontsize=fs - 1,
        )
        if i_d == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=axs, label=cbar_label, shrink=0.7, aspect=30, pad=0.02,
)
fig.tight_layout(pad=0.4)
plotter.save_or_show(fig, save_path=str(save_folder / "localisation_precision_surface.svg"))

In [ ]:
# ── Figure 3: colour precision — surface map ──────────────────────────────────
# Same layout: 2 rows × 3 cols. Shared colour scale across all dyes and depths.
metric     = "colour_dist"
cbar_label = r"$\sigma_{\mathrm{colour}}$ (a.u.)"

data_m = da_defocus.sel(metric=metric).values
vmin, vmax = float(np.nanmin(data_m)), float(np.nanmax(data_m))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(
    nrows=len(coverslip_depths_nm), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
    squeeze=False,
)
for i_d, d_nm in enumerate(coverslip_depths_nm):
    for i_c, dye in enumerate(dyes):
        ax = axs[i_d, i_c]
        Z = da_defocus.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel(
            f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "",
            fontsize=fs - 1,
        )
        if i_d == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=axs, label=cbar_label, shrink=0.7, aspect=30, pad=0.02,
)
fig.tight_layout(pad=0.4)
plotter.save_or_show(fig, save_path=str(save_folder / "colour_precision_surface.svg"))

In [ ]:
# ── Figure 4: combined summary — 2 metrics × 3 dyes, independent per-row scale ─
# Each row uses its own colour normalisation so both metrics are maximally readable.
# Coverslip depth = 0 nm (no spherical aberration) for the primary summary view.
metric_list  = ["sigma_xy",            "colour_dist"]
cbar_labels4 = [r"$\sigma_{xy}$ / nm", r"$\sigma_{\mathrm{colour}}$ (a.u.)"]
i_d_summary  = 0   # coverslip_nm index 0 → d = 0 nm

fig, axs = plt.subplots(
    nrows=len(metric_list), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(metric_list)),
    squeeze=False,
)
for i_m, (metric, cbar_lbl) in enumerate(zip(metric_list, cbar_labels4)):
    # Normalise across all dyes at this coverslip depth
    data_row = da_defocus.isel(coverslip_nm=i_d_summary).sel(metric=metric).values
    vmin, vmax = float(np.nanmin(data_row)), float(np.nanmax(data_row))
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for i_c, dye in enumerate(dyes):
        ax = axs[i_m, i_c]
        Z = da_defocus.isel(dye=i_c, coverslip_nm=i_d_summary).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel("Photons" if i_c == 0 else "", fontsize=fs - 1)
        if i_m == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

    fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
        ax=axs[i_m, :].tolist(),
        label=cbar_lbl, shrink=0.85, aspect=20, pad=0.02,
    )

fig.suptitle(f"d = 0 nm (at coverslip)", fontsize=fs, y=1.01)
fig.tight_layout(pad=0.4)
plotter.save_or_show(fig, save_path=str(save_folder / "combined_summary_surface.svg"))